# Sensitivity to feature perturbations

Each CSV row describes a flow. Swapping columns can preserve tensor shape while changing what each input means.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# ── Run configuration ──
# The seed identifies the random choices in this run. A comparison across methods also
# depends on using the same training and test examples.
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPSILONS = [0.0, 0.01, 0.05, 0.1, 0.2, 0.5, 0.8, 1.0]
EVALUATION_PGD_ITERATIONS = 5
PGD_ALPHA_FRACTION = 0.1
NOISE_REPEATS = 3
BATCH_SIZE = 256
MAX_TEST_SAMPLES = None
MODEL_NAMES = ["MLP", "RNN", "CNN", "GRU", "LSTM"]

EXPORT_FIELD_NAMES = {
    "caso": "scenario", "modelo": "model", "tipo": "task", "ataque": "attack",
    "amostras": "samples", "accuracy_limpa": "clean_accuracy", "macro_f1_limpo": "clean_macro_f1",
    "origem_modelos": "model_source", "configuracao": "configuration", "metricas_limpas": "clean_metrics",
    "resultados_ataques": "attack_results",
    "dados_binarys_cleaneds": "binary_cleaned_data",
    "dados_multiclass_cleaneds": "multiclass_cleaned_data",
    "dados_binarios_tratados": "binary_cleaned_data", "dados_multiclasse_tratados": "multiclass_cleaned_data",
}
EXPORT_VALUE_NAMES = {
    "binario": "binary", "multiclasse": "multiclass", "ruido_gaussiano": "gaussian_noise",
    "classificacao_binaria_cleaned": "treated_binary_classification",
    "classificacao_multiclasse_cleaned": "treated_multiclass_classification",
    "binario_original": "binary_original", "binario_tratado": "binary_cleaned",
    "multiclasse_original": "multiclass_original", "multiclasse_tratado": "multiclass_cleaned",
    "StandardScaler; epsilon/sigma em unidades do espaco padronizado":
        "StandardScaler; epsilon/sigma in standardized-space units",
}

def translate_export(value):
    if isinstance(value, dict):
        return {EXPORT_VALUE_NAMES.get(key, EXPORT_FIELD_NAMES.get(key, key)): translate_export(item) for key, item in value.items()}
    if isinstance(value, list):
        return [translate_export(item) for item in value]
    if isinstance(value, str):
        return EXPORT_VALUE_NAMES.get(value, value)
    return value

EXECUTION_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = (
    EXECUTION_DIR,
    EXECUTION_DIR / "deep-learning-models",
    EXECUTION_DIR.parent / "deep-learning-models",
)
ROOT = next((path for path in ROOT_CANDIDATES if (path / "Normal+Attacks.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError(f"Could not locate Normal+Attacks.csv from {EXECUTION_DIR}")
ROOT = ROOT.resolve()
PROJECT_DIR = ROOT.parent
DATASET_PATH = ROOT / "Normal+Attacks.csv"
ORIGINAL_MODELS_DIR = ROOT / "trained_models"
RESULTS_DIR = PROJECT_DIR / "adversarial-attacks" / "adversarial_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = PROJECT_DIR / "graphics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def locate_cleaned_data(directory, task):
    root_path = ROOT / directory
    summary_path = root_path / "training_summary.json"
    models_dir = root_path / "models"
    scaler = "scaler_binary.joblib" if task == "binary" else "scaler_multiclass.joblib"
    model_files = [models_dir / f"{name}_{task}.pt" for name in MODEL_NAMES]
    if not summary_path.exists() or not (models_dir / scaler).exists() or not all(p.exists() for p in model_files):
        raise FileNotFoundError(f"Current classification results for {task} were not found or are incomplete in {root_path}")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if task == "multiclass" and summary.get("train_test_group_overlap") != 0:
        raise ValueError(f"Multiclass classification does not confirm disjoint groups in {summary_path}")
    return root_path


BINARY_CLEANED_DIR = locate_cleaned_data("treated_binary_classification", "binary")
MULTICLASS_CLEANED_DIR = locate_cleaned_data("treated_multiclass_classification", "multiclass")
BINARY_HYPERPARAMETERS = json.loads((ROOT / "bayesian_tuning" / "best_hyperparameters_summary_binary.json").read_text(encoding="utf-8"))
MULTICLASS_HYPERPARAMETERS = json.loads((ROOT / "bayesian_tuning" / "best_hyperparameters_summary_multiclass.json").read_text(encoding="utf-8"))

CASES = {
    "binary_original": {
        "task": "binary", "use_cleaned_data": False,
        "model_dir": ORIGINAL_MODELS_DIR, "scaler_path": ORIGINAL_MODELS_DIR / "scaler_binary.joblib",
        "hp": BINARY_HYPERPARAMETERS,
    },
    "binary_cleaned": {
        "task": "binary", "use_cleaned_data": True,
        "model_dir": BINARY_CLEANED_DIR / "models", "scaler_path": BINARY_CLEANED_DIR / "models" / "scaler_binary.joblib",
        "hp": BINARY_HYPERPARAMETERS,
    },
    "multiclass_original": {
        "task": "multiclass", "use_cleaned_data": False,
        "model_dir": ORIGINAL_MODELS_DIR, "scaler_path": ORIGINAL_MODELS_DIR / "scaler_multiclass.joblib",
        "hp": MULTICLASS_HYPERPARAMETERS,
    },
    "multiclass_cleaned": {
        "task": "multiclass", "use_cleaned_data": True,
        "model_dir": MULTICLASS_CLEANED_DIR / "models", "scaler_path": MULTICLASS_CLEANED_DIR / "models" / "scaler_multiclass.joblib",
        "hp": MULTICLASS_HYPERPARAMETERS,
    },
}


def prepare_case(key, config):
    df = pd.read_csv(DATASET_PATH)
    scaler = joblib.load(config["scaler_path"])

    if not config["use_cleaned_data"]:
        X = df.drop(columns=["label"]).select_dtypes(include="number")
        if config["task"] == "binary":
            y = df["label"].astype(str).str.strip().str.lower().ne("normal").astype("int64").to_numpy()
            class_names = ["Normal", "Attack"]
        else:
            encoder = joblib.load(config["model_dir"] / "label_encoder_multiclass.joblib")
            y = encoder.transform(df["label"])
            class_names = encoder.classes_.tolist()
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=SEED, stratify=y
        )
    else:
        df = df.copy()
        if config["task"] == "binary":
            df["target"] = df["label"].astype(str).str.strip().str.lower().ne("normal").astype("int64")
        else:
            df["label"] = df["label"].replace({"Reco": "Reco+Scanning", "Scanning": "Reco+Scanning"})
            df["target"] = df["label"]
        df = df.drop(columns=["fwd_seg_size_min"], errors="ignore")
        all_features = df.drop(columns=["label", "target"], errors="ignore").select_dtypes(include="number")
        feature_names = all_features.columns.tolist()
        data = all_features.copy()
        data["target"] = df["target"].to_numpy()

        # ── Duplicate rows ──
        # The target is already in the table. Deduplication compares complete records.
        # Grouping later handles identical feature vectors retained because their labels
        # differ.
        data = data.drop_duplicates().reset_index(drop=True)
        X = data[feature_names]
        if config["task"] == "binary":
            y = data["target"].to_numpy(dtype="int64")
            class_names = ["Normal", "Attack"]

            # ── Groups for splitting ──
            # The hash depends on features and ignores the row index. Copies at different
            # positions receive the same group key.
            groups = pd.util.hash_pandas_object(X, index=False).to_numpy()
            splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
        else:
            encoder = joblib.load(config["model_dir"] / "label_encoder_multiclass.joblib")
            y = encoder.transform(data["target"])
            class_names = encoder.classes_.tolist()

            # ── Groups for splitting ──
            # The hash depends on features and ignores the row index. Copies at different
            # positions receive the same group key.
            groups = pd.util.hash_pandas_object(X, index=False).to_numpy()
            splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
        train_indices, test_indices = next(splitter.split(X, y, groups=groups))
        X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
        y_train, y_test = y[train_indices], y[test_indices]

    scaler_feature_names = list(scaler.feature_names_in_)
    X_test = X_test.loc[:, scaler_feature_names]
    X_test_np = scaler.transform(X_test).astype(np.float32)
    y_test_np = np.asarray(y_test, dtype=np.int64)

    if MAX_TEST_SAMPLES is not None and len(y_test_np) > MAX_TEST_SAMPLES:
        if MAX_TEST_SAMPLES < len(np.unique(y_test_np)):
            raise ValueError("MAX_TEST_SAMPLES must be at least the number of test classes")
        _, indices = train_test_split(
            np.arange(len(y_test_np)), train_size=MAX_TEST_SAMPLES,
            random_state=SEED, stratify=y_test_np,
        )
        X_test_np, y_test_np = X_test_np[indices], y_test_np[indices]

    return X_test_np, y_test_np, class_names, scaler_feature_names


class MLPModel(nn.Module):
    def __init__(self, n_features, hp, n_outputs):
        super().__init__()
        layers = []
        previous = n_features
        for index in range(hp["n_camadas"]):
            units = hp[f"unidades_{index}"]
            layers.extend([nn.Linear(previous, units), nn.ReLU() if hp["ativacao"] == "relu" else nn.Tanh()])
            dropout = hp[f"dropout_{index}"]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous = units
        layers.append(nn.Linear(previous, n_outputs))
        self.rede = nn.Sequential(*layers)

    def forward(self, x):
        return self.rede(x)


# ── Recurrent layers ──
# The traversed axis contains the features of one flow. Column order therefore defines the
# sequence seen by the network.
class RecurrentModel(nn.Module):
    def __init__(self, architecture, hp, n_outputs):
        super().__init__()
        self.camadas = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        previous = 1
        recurrent_types = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}
        for index in range(hp["n_camadas"]):
            units = hp[f"unidades_{index}"]
            self.camadas.append(recurrent_types[architecture](previous, units, batch_first=True))
            dropout = hp[f"dropout_{index}"]
            self.dropouts.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
            previous = units
        self.densa = nn.Linear(previous, 64)
        self.saida = nn.Linear(64, n_outputs)

    def forward(self, x):
        for index, (layer, dropout) in enumerate(zip(self.camadas, self.dropouts)):
            output, _ = layer(x)
            output = dropout(output)
            x = output if index < len(self.camadas) - 1 else output[:, -1, :]
        return self.saida(torch.relu(self.densa(x)))


# ── Convolution over features ──
# The input has one channel. A filter combines neighboring features in CSV order, so column
# position matters.
class CNNModel(nn.Module):
    def __init__(self, hp, n_outputs):
        super().__init__()
        layers = []
        previous = 1
        kernel_size = hp["kernel_size"]
        for index in range(hp["n_blocos_conv"]):
            filters = hp[f"filtros_{index}"]
            layers.extend([nn.Conv1d(previous, filters, kernel_size, padding=kernel_size // 2), nn.ReLU(), nn.MaxPool1d(2, ceil_mode=True)])
            previous = filters
        self.conv = nn.Sequential(*layers)
        self.pool_global = nn.AdaptiveAvgPool1d(1)
        self.densa = nn.Linear(previous, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(hp.get("dropout_final", 0.2))
        self.saida = nn.Linear(128, n_outputs)

    def forward(self, x):
        x = self.pool_global(self.conv(x)).squeeze(-1)
        return self.saida(self.dropout(self.relu(self.densa(x))))


def create_model(name, config, checkpoint):
    hp = config["hp"][name]["hiperparametros"]
    is_binary = config["task"] == "binary"
    n_outputs = 1 if is_binary else int(checkpoint["n_classes"])
    n_features = int(checkpoint["n_features"])
    if name == "MLP":
        model = MLPModel(n_features, hp, n_outputs)
    elif name == "CNN":
        model = CNNModel(hp, n_outputs)
    else:
        model = RecurrentModel(name, hp, n_outputs)
    model.load_state_dict(checkpoint["state_dict"])
    model.to(DEVICE).eval()
    return model


print(f"Device: {DEVICE}")
print(f"Cleaned binary data: {BINARY_CLEANED_DIR}")
print(f"Cleaned multiclass data: {MULTICLASS_CLEANED_DIR}")
print(f"Current results: {RESULTS_DIR}")

The comparison traverses models and evaluation conditions. Perturbations here apply to previously extracted numeric features.

In [ ]:
from contextlib import contextmanager



# The input batch is a table of flows. The extra dimension occupies a different
# position in CNN and recurrent layers because their input conventions differ.
def prepare_input(x, name):
    if name in ("RNN", "GRU", "LSTM"):
        return x.unsqueeze(-1)
    if name == "CNN":
        return x.unsqueeze(-1).permute(0, 2, 1)
    return x


def model_loss(logits, y, is_binary):
    if is_binary:
        return nn.functional.binary_cross_entropy_with_logits(logits.view(-1), y.float())
    return nn.functional.cross_entropy(logits, y.long())


def predict(model, name, X_np, is_binary):
    predictions = []
    probabilities = []
    with torch.no_grad():
        for start_index in range(0, len(X_np), BATCH_SIZE):
            xb = torch.as_tensor(X_np[start_index:start_index + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
            logits = model(prepare_input(xb, name))
            if is_binary:
                probs = torch.sigmoid(logits.view(-1))
                probabilities.extend(probs.cpu().numpy().tolist())
                predictions.extend((probs >= 0.5).long().cpu().numpy().tolist())
            else:
                predictions.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
    return np.asarray(predictions, dtype=np.int64), np.asarray(probabilities, dtype=np.float32)


def evaluate_metrics(y_true, y_pred, class_names, is_binary):
    labels = np.arange(len(class_names))
    matriz = confusion_matrix(y_true, y_pred, labels=labels)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "classification_report": classification_report(
            y_true, y_pred, labels=labels, target_names=class_names,
            output_dict=True, zero_division=0,
        ),
        "confusion_matrix": matriz.tolist(),
    }
    if is_binary:
        tn, fp, fn, tp = matriz.ravel()
        metrics.update({
            "false_positive_rate": float(fp / (fp + tn)) if fp + tn else 0.0,
            "specificity": float(tn / (tn + fp)) if tn + fp else 0.0,
            "attack_recall": float(tp / (tp + fn)) if tp + fn else 0.0,
            "attack_precision": float(tp / (tp + fp)) if tp + fp else 0.0,
            "true_negative": int(tn), "false_positive": int(fp),
            "false_negative": int(fn), "true_positive": int(tp),
        })
    return metrics


@contextmanager
def modo_gradiente(model):
    model.train()
    for modulo in model.modules():
        if isinstance(modulo, nn.Dropout):
            modulo.eval()
    try:
        yield
    finally:
        model.eval()


def generate_fgsm(model, name, xb, y_batch, is_binary, epsilon):
    if epsilon == 0:
        return xb.detach()
    with modo_gradiente(model):
        input_gradient = xb.detach().requires_grad_(True)
        logits = model(prepare_input(input_gradient, name))
        loss = model_loss(logits, y_batch, is_binary)
        gradient = torch.autograd.grad(loss, input_gradient)[0]
        return (input_gradient + epsilon * gradient.sign()).detach()


def generate_pgd(model, name, xb, y_batch, is_binary, epsilon):
    if epsilon == 0:
        return xb.detach()
    x_original = xb.detach()
    alpha = max(epsilon * PGD_ALPHA_FRACTION, 1e-6)
    x_adv = x_original + torch.empty_like(x_original).uniform_(-epsilon, epsilon)
    with modo_gradiente(model):
        for _ in range(EVALUATION_PGD_ITERATIONS):
            x_adv = x_adv.detach().requires_grad_(True)
            logits = model(prepare_input(x_adv, name))
            loss = model_loss(logits, y_batch, is_binary)
            gradient = torch.autograd.grad(loss, x_adv)[0]
            with torch.no_grad():
                x_adv = x_adv + alpha * gradient.sign()
                x_adv = torch.maximum(torch.minimum(x_adv, x_original + epsilon), x_original - epsilon)
    return x_adv.detach()


def apply_attack(model, name, X_np, y_np, is_binary, epsilon, attack, seed):
    torch.manual_seed(seed)
    results = []
    for start_index in range(0, len(X_np), BATCH_SIZE):
        xb = torch.as_tensor(X_np[start_index:start_index + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
        y_batch = torch.as_tensor(y_np[start_index:start_index + BATCH_SIZE], dtype=torch.long, device=DEVICE)
        if attack == "fgsm":
            x_adv = generate_fgsm(model, name, xb, y_batch, is_binary, epsilon)
        elif attack == "pgd":
            x_adv = generate_pgd(model, name, xb, y_batch, is_binary, epsilon)
        else:
            x_adv = xb + epsilon * torch.randn_like(xb)
        results.append(x_adv.detach().cpu().numpy())
    return np.concatenate(results, axis=0)


def load_checkpoint(case_config, name):
    suffix = "binary" if case_config["task"] == "binary" else "multiclass"
    path = case_config["model_dir"] / f"{name}_{suffix}.pt"
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    scaler = joblib.load(case_config["scaler_path"])
    if int(checkpoint["n_features"]) != int(scaler.n_features_in_):
        raise ValueError(f"Checkpoint and scaler features do not match: {path}")
    model = create_model(name, case_config, checkpoint)
    return model, checkpoint


def evaluate_model(case_name, case_config, name, X_np, y_np, class_names, epsilons=EPSILONS):
    model, checkpoint = load_checkpoint(case_config, name)
    is_binary = case_config["task"] == "binary"
    clean_predictions, _ = predict(model, name, X_np, is_binary)
    results = []
    clean_metrics = evaluate_metrics(y_np, clean_predictions, class_names, is_binary)

    for epsilon_index, epsilon in enumerate(epsilons):
        for attack_index, attack in enumerate(("fgsm", "pgd", "ruido_gaussiano")):
            repetitions = NOISE_REPEATS if attack == "ruido_gaussiano" and epsilon > 0 else 1
            repeated_metrics = []
            for repetition in range(repetitions):
                attack_seed = SEED + epsilon_index * 1000 + attack_index * 100 + repetition
                X_adversarial = apply_attack(
                    model, name, X_np, y_np, is_binary, epsilon, attack, attack_seed
                )
                y_pred, _ = predict(model, name, X_adversarial, is_binary)
                repeated_metrics.append(evaluate_metrics(y_np, y_pred, class_names, is_binary))

            row = {
                "caso": case_name,
                "modelo": name,
                "tipo": case_config["task"],
                "ataque": attack,
                "epsilon": float(epsilon),
                "amostras": int(len(y_np)),
                "accuracy_limpa": clean_metrics["accuracy"],
                "macro_f1_limpo": clean_metrics["macro_f1"],
            }
            for key in ("accuracy", "macro_f1", "false_positive_rate", "specificity", "attack_recall", "attack_precision"):
                values = [m[key] for m in repeated_metrics if key in m]
                if values:
                    row[key] = float(np.mean(values))
                    if len(values) > 1:
                        row[f"{key}_std"] = float(np.std(values))
            results.append(row)

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return results, clean_metrics


print("Inference functions, attacks, and metrics are ready.")

## Data and models

Metrics, model and preprocessing belong together. When reusing a checkpoint, load its matching scaler and, for multiclass tasks, check the label order in the encoder.

In [ ]:
# ── Scenario reference ──
# The same test rows recur in every condition of a scenario. Keeping this reference also
# avoids rebuilding the dataset for each evaluation.
CASE_DATA = {}
CLEAN_METRICS = {}

for case_name, case_config in CASES.items():
    X_test, y_test, class_names, feature_names = prepare_case(case_name, case_config)
    CASE_DATA[case_name] = {
        "X": X_test,
        "y": y_test,
        "class_names": class_names,
        "feature_names": feature_names,
    }
    CLEAN_METRICS[case_name] = {}
    print(f"\n{case_name}: {len(y_test):,} samples, {len(feature_names)} features")
    print("Test classes:", dict(zip(*np.unique(y_test, return_counts=True))))

    for name in MODEL_NAMES:
        model, checkpoint = load_checkpoint(case_config, name)
        y_pred, _ = predict(model, name, X_test, case_config["task"] == "binary")
        metrics = evaluate_metrics(y_test, y_pred, class_names, case_config["task"] == "binary")
        CLEAN_METRICS[case_name][name] = metrics
        print(
            f"  {name:5s} | accuracy={metrics['accuracy']:.4f} "
            f"| F1 macro={metrics['macro_f1']:.4f}"
        )
        del model

print("\nClean evaluation completed. No attack was run in this cell.")

This check uses a few examples to catch shape or gradient incompatibilities before the full evaluation. A valid output here allows the dataset evaluation to proceed.

In [ ]:
# ── Small-batch check ──
# This small batch exposes mismatches between input dimensions and architecture before the
# full evaluation loop.
SMOKE_TEST_SAMPLES = 8
SMOKE_TEST_EPSILON = 0.01

for case_name, case_config in CASES.items():
    data = CASE_DATA[case_name]
    X_smoke = data["X"][:SMOKE_TEST_SAMPLES]
    y_smoke = data["y"][:SMOKE_TEST_SAMPLES]
    for name in MODEL_NAMES:
        model, _ = load_checkpoint(case_config, name)
        is_binary = case_config["task"] == "binary"
        adversariais = {}
        for attack in ("fgsm", "pgd", "ruido_gaussiano"):
            adversariais[attack] = apply_attack(
                model, name, X_smoke, y_smoke, is_binary,
                SMOKE_TEST_EPSILON, attack, SEED,
            )
            assert adversariais[attack].shape == X_smoke.shape
            assert np.isfinite(adversariais[attack]).all()
        del model
    print(f"{case_name}: as 5 arquiteturas passaram FGSM, PGD e ruido.")

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print("Quick check completed; the final cell still runs the full evaluation.")

## Evaluation metrics

Accuracy counts all correct predictions. Macro-F1 gives equal weight to each class F1. For ASR, align clean and perturbed predictions from the same model by row. If it originally got 80 examples right and now misses 8 of those, ASR is 8/80 = 10%.

In [ ]:
RESULTS = []
for case_index, (case_name, case_config) in enumerate(CASES.items()):
    data = CASE_DATA[case_name]
    print(f"\nRunning attacks: {case_name}")
    for model_index, name in enumerate(tqdm(MODEL_NAMES, desc=case_name)):
        rows, _ = evaluate_model(
            case_name,
            case_config,
            name,
            data["X"],
            data["y"],
            data["class_names"],
        )
        for row in rows:
            row["origem_modelos"] = (
                "treated_binary_classification" if case_name == "binary_cleaned"
                else "treated_multiclass_classification" if case_name == "multiclass_cleaned"
                else "original"
            )
        RESULTS.extend(rows)

# ── Results table ──
# Each row holds metrics for one condition. The plots below select rows by scenario before
# drawing the curves.
results_df = pd.DataFrame(RESULTS)
csv_path = RESULTS_DIR / "attack_metrics.csv"
exported_results = results_df.rename(columns=EXPORT_FIELD_NAMES).replace(EXPORT_VALUE_NAMES)
exported_results.to_csv(csv_path, index=False)

relatorio_json = {
    "configuracao": {
        "seed": SEED,
        "epsilons": EPSILONS,
        "pgd_iterations": EVALUATION_PGD_ITERATIONS,
        "pgd_alpha_fraction": PGD_ALPHA_FRACTION,
        "gaussian_repeats": NOISE_REPEATS,
        "batch_size": BATCH_SIZE,
        "max_test_samples": MAX_TEST_SAMPLES,
        "feature_space": "StandardScaler; epsilon/sigma em unidades do espaco padronizado",
        "binary_threshold": 0.5,
        "dados_binarys_cleaneds": "treated_binary_classification",
        "dados_multiclass_cleaneds": "treated_multiclass_classification",
    },
    "clean_metrics": CLEAN_METRICS,
    "attack_results": RESULTS,
}
json_path = RESULTS_DIR / "attack_metrics.json"
json_path.write_text(json.dumps(translate_export(relatorio_json), indent=2, ensure_ascii=False), encoding="utf-8")

for case_name, case_config in CASES.items():
    case_data = results_df[results_df["caso"] == case_name]
    fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharex=True)
    for axis, attack in zip(axes, ("fgsm", "pgd", "ruido_gaussiano")):
        subset = case_data[case_data["ataque"] == attack]
        for name in MODEL_NAMES:
            series = subset[subset["modelo"] == name].sort_values("epsilon")
            axis.plot(series["epsilon"], series["accuracy"], marker="o", label=name)
        axis.set_title({"fgsm": "FGSM", "pgd": "PGD", "ruido_gaussiano": "Gaussian noise"}[attack])
        axis.set_xlabel("Epsilon")
        axis.set_ylabel("Accuracy")
        axis.set_ylim(0, 1.02)
        axis.grid(alpha=0.3)
    axes[-1].legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.suptitle(f"Adversarial robustness: {case_name.replace('_', ' ').title()}")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"robustness_{case_name}_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    if case_config["task"] == "binary":
        fig, axis = plt.subplots(figsize=(8, 5))
        subset = case_data[case_data["ataque"].isin(["fgsm", "pgd", "ruido_gaussiano"])]
        for attack in ("fgsm", "pgd", "ruido_gaussiano"):
            by_attack = subset[subset["ataque"] == attack]
            for name in MODEL_NAMES:
                series = by_attack[by_attack["modelo"] == name].sort_values("epsilon")
                axis.plot(series["epsilon"], series["false_positive_rate"], marker="o", label=f"{name} / {attack}")
        axis.set_title(f"False-positive rate: {case_name.replace('_', ' ').title()}")
        axis.set_xlabel("Epsilon")
        axis.set_ylabel("FPR among Normal examples")
        axis.set_ylim(0, 1.02)
        axis.grid(alpha=0.3)
        axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", ncol=1)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / f"robustness_{case_name}_false_positive_rate.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)

RESULTS_RELATIVE_DIR = Path("adversarial_results")
print(f"\nMetrics CSV: {RESULTS_RELATIVE_DIR / csv_path.name}")
print(f"Metrics JSON: {RESULTS_RELATIVE_DIR / json_path.name}")
print(f"Figures and files from this run: {RESULTS_RELATIVE_DIR}")